# 04 — Ensemble

Span voting: keep a location found by ≥ k of N models.
Models: fine-tuned M6 variants (`data/preds/`). Evaluation: holdout, sample, synth2.

### Imports

In [1]:
import ast
import json
import re
import string
from collections import Counter
from typing import List

import pandas as pd

DATA = '../data'
MODELS = ['M6s_finetune', 'M6s_ru', 'M6s_ru_vocab', 'M6s_5k']

### Competition metric (Kaggle, unchanged)

In [2]:
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()


def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p, = 0.0, 0.0, 0.0, 0.0
    for y_true_sample, y_pred_sample in zip(y_true, y_pred):
        y_true_sample = set([hard_processing(s) for s in y_true_sample])
        y_pred_sample = set([hard_processing(s) for s in y_pred_sample])
        tp += len(y_true_sample & y_pred_sample)
        fp += len(y_pred_sample - y_true_sample)
        fn += len(y_true_sample - y_pred_sample)
        p += len(y_true_sample)
    if tp + fp == 0:
        precision = 1.0 if p == 0 else 0.0
    else:
        precision = tp / (tp + fp)
    if tp + fn == 0:
        recall = 1.0 if p == 0 else 0.0
    else:
        recall = tp / (tp + fn)
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return round(f1, 8)

### Gold and predictions

In [3]:
uk = pd.read_csv(f'{DATA}/uk_geo_dataset.csv')
split = pd.read_csv(f'{DATA}/splits/split.csv')
holdout = uk.loc[split.loc[split['split'] == 'holdout', 'row_id']]
sample = pd.read_csv(f'{DATA}/labeling_sample.csv')
synth2 = pd.read_json(f'{DATA}/synthetic_posts_2.jsonl', lines=True)
test = pd.read_csv(f'{DATA}/test.csv')

GOLD = {
    'holdout': [[t[s:e] for s, e in ast.literal_eval(m)] for t, m in zip(holdout['text'], holdout['loc_markers'])],
    'sample': list(sample['locations'].map(ast.literal_eval)),
    'synth2': list(synth2['locations']),
}
preds = {name: json.load(open(f'{DATA}/preds/{name}.json')) for name in MODELS}
assert all(len(preds[m]['holdout']) == len(GOLD['holdout']) for m in MODELS)

### Single models

In [4]:
def scores(set_preds):
    return {name: round(comp_metric(GOLD[name], set_preds[name]), 4) for name in GOLD}


single = pd.DataFrame({m: scores(preds[m]) for m in MODELS}).T
single['LB'] = [0.60677, 0.61904, 0.62368, 0.61426]
single

,holdout,sample,synth2,LB
M6s_finetune,0.8896,0.7692,0.8447,0.60677
M6s_ru,0.8890,0.6757,0.8231,0.61904
M6s_ru_vocab,0.8895,0.6944,0.8336,0.62368
M6s_5k,0.8837,0.7500,0.9913,0.61426


### Voting
M6s_5k was trained on synth2, so k is chosen on the 3-model ensemble without it, then applied to all 4.

In [5]:
def vote(models, set_name, k):
    result = []
    for posts in zip(*[preds[m][set_name] for m in models]):
        counts = Counter(loc for locs in posts for loc in set(locs))
        order = list(dict.fromkeys(loc for locs in posts for loc in locs))
        result.append([loc for loc in order if counts[loc] >= k])
    return result


def vote_scores(models, k):
    return scores({name: vote(models, name, k) for name in GOLD})


without_5k = MODELS[:3]
pd.DataFrame({f'3 models, k={k}': vote_scores(without_5k, k) for k in [1, 2, 3]}
             | {f'4 models, k={k}': vote_scores(MODELS, k) for k in [1, 2, 3, 4]}).T

,holdout,sample,synth2
"3 models, k=1",0.8864,0.7500,0.8493
"3 models, k=2",0.8907,0.6944,0.8300
"3 models, k=3",0.8909,0.6944,0.8211
"4 models, k=1",0.8833,0.7143,0.9205
"4 models, k=2",0.8892,0.7792,0.8725
"4 models, k=3",0.8904,0.6944,0.8412
"4 models, k=4",0.8887,0.7042,0.8559


### Submission

In [6]:
K = 2
ensemble_test = vote(MODELS, 'test', K)
pd.DataFrame({'text_id': test['text_id'],
              'locations': [json.dumps(locs, ensure_ascii=False) for locs in ensemble_test]}) \
    .to_csv(f'{DATA}/submissions/ensemble_vote.csv', index=False)
print('Test spans:', sum(map(len, ensemble_test)))

Test spans: 775


## Conclusion
- synth2 and LB disagree across models (M6s_finetune: best synth2, worst LB): synth2 compares post-processing rules of one model, not different models.
- Voting gives a small gain: 3 models, k=1 → synth2 0.849 (+0.5 over the best single model), holdout unchanged.
- Final: 4 models, k=2. Not verified on LB (daily limit reached).